# Import libraries

In [ ]:
import torch
from transformers import AutoTokenizer
from TweetNormalizer import normalizeTweet
import pandas as pd
import torch.nn as nn
from transformers import AutoModel
import pickle

# Prepare inputs and labels

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [ ]:
def create_tweets_embeddings(tweets):
    tweets = [normalizeTweet(tweet) for tweet in tweets]
    tweets = tokenizer(tweets, padding=True, truncation=True, return_tensors="pt")
    return tweets["input_ids"].clone().detach(), tweets["attention_mask"]

In [ ]:
def create_label_encodings(labels): #change function name
    # change labels to numbers
    # 0: Demonstrations, 1: Political violence, 2: No relevant event, 3: Strategic developments, 4: Battles, 5: Strategic developments, 6: Riots
    labels = [0 if label == "Protests" else
            1 if label == "Violence against civilians" else
            2 if label == "No relevant event" else
            3 if label == "Explosions/Remote violence" else
            4 if label == "Battles" else
            5 if label == "Strategic developments" else
            6 if label == "Riots" else
            None for label in labels]

    # create tensor of labels
    labels = torch.tensor(labels, dtype=torch.long)

    return labels

In [ ]:
def features_to_tensor(features):
    return torch.tensor(features, dtype=torch.float32)

In [ ]:
def dataset_to_embeddings(file_path):
    df = pd.read_csv(file_path)
    tweets = df["text"].tolist()
    labels = df["event_type"].tolist()
    features = df[["sentiment_pos","sentiment_neg","hs_hateful","hs_targeted","hs_aggressive","emo_anger",
                   "emo_fear","emo_disgust","emo_sadness","emo_surprise","sentiment_neu"]].values.tolist()

    tweets_embeddings, attention_mask = create_tweets_embeddings(tweets)
    labels = create_label_encodings(labels) #change function name
    features = features_to_tensor(features).float()

    return (tweets_embeddings, features, attention_mask), labels

In [ ]:
# test function
inputs, labels = dataset_to_embeddings("train.csv")
text_input_ids = inputs[0]
numerical_features = inputs[1]
attention_masks = inputs[2]

In [ ]:
labels[:20]

tensor([0, 1, 1, 3, 0, 1, 4, 1, 0, 1, 5, 3, 0, 0, 0, 0, 1, 2, 1, 0])

## Create embeddings

In [ ]:
bertweet = AutoModel.from_pretrained("distilbert/distilbert-base-uncased")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [ ]:
def create_numerical_embeddings(numerical_feature_dim, hidden_dim, numerical_features):
    fc_numerical = nn.Sequential(
      nn.Linear(numerical_feature_dim, hidden_dim),
      nn.ReLU(),
      nn.Dropout(0.3)
    )

    embeddings = fc_numerical(numerical_features)

    return embeddings

In [ ]:
# Get text embeddings from BERTweet
text_embeddings = bertweet(input_ids=text_input_ids, attention_mask=attention_masks).last_hidden_state

# Pool the embeddings (use the [CLS] token representation or mean pooling)
text_embeddings = text_embeddings.mean(dim=1)

In [ ]:
hidden_dim = 128
numerical_feature_dim = 11

# Process numerical features
numerical_embeddings = create_numerical_embeddings(numerical_feature_dim, hidden_dim, numerical_features)

In [ ]:
# Concatenate text and numerical embeddings
combined_embeddings = torch.cat((text_embeddings, numerical_embeddings), dim=1)
print(combined_embeddings.shape)

torch.Size([4919, 896])


# Custom Dataset

In [ ]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs
        self.labels = labels

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return {
            'inputs': self.inputs[idx],
            'labels': self.labels[idx]
        }

# Neural Network

In [ ]:
class CombinedModel(nn.Module):
    def __init__(self, hidden_dim, output_dim, input_dim, num_hidden_layers):
        '''
        A simple model that combines text embeddings from BERTweet with numerical features to predict the disorder type.

        Args:
            # text_embedding_dim (int): Dimension of the text embeddings from BERTweet. remove
            numerical_feature_dim (int): Dimension of the numerical features. remove
            hidden_dim (int): Dimension of the hidden layers.
            output_dim (int): Dimension of the output layer.
        '''
        super(CombinedModel, self).__init__()

        # Fully connected layers for combined features
        self.fc_combined = nn.Sequential()
        self.fc_combined.append(nn.Linear(input_dim, hidden_dim))
        self.fc_combined.append(nn.ReLU())
        self.fc_combined.append(nn.Dropout(0.3))
        for _ in range(num_hidden_layers - 1):  # Subtract 1 to account for the first layer
            self.fc_combined.append(nn.Linear(hidden_dim, hidden_dim))
            self.fc_combined.append(nn.ReLU())
            self.fc_combined.append(nn.Dropout(0.3))
        self.fc_combined.append(nn.Linear(hidden_dim, output_dim))  # Output layer

    def forward(self, combined_embeddings):
        '''
        Forward pass of the model.
        '''
        # Process combined embeddings
        output = self.fc_combined(combined_embeddings)

        return output

In [ ]:
# Define model parameters
hidden_dim = 128
input_dim = combined_embeddings.shape[1]  # Number of input features
output_dim = 8  # Number of classes for disorder type classification

# Training model

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from torch.utils.data import TensorDataset
import torch.nn.functional as F

In [ ]:
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score

In [ ]:
class Estimator(BaseEstimator, TransformerMixin):
    def __init__(self, input_dim=896,
                 hidden_dim=128, output_dim=8, num_epochs=10, lr=0.001, batch_size=32, num_hidden_layers=2):
      # consider adding number of hidden layers
      self.input_dim = input_dim
      self.hidden_dim = hidden_dim
      self.output_dim = output_dim
      self.num_epochs = num_epochs
      self.lr = lr
      self.batch_size = batch_size
      self.num_hidden_layers = num_hidden_layers
      self.model = CombinedModel(self.hidden_dim, self.output_dim, self.input_dim, self.num_hidden_layers)
      self.criterion = nn.CrossEntropyLoss()
      self.optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)

    def fit(self, X, y):
      X = torch.tensor(X)
      y = torch.tensor(y)
      dataset = TensorDataset(X, y)
      dataloader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
      # train_model(self.model, self.criterion, self.optimizer, dataloader, self.num_epochs)
      for epoch in range(self.num_epochs):
        self.model.train()
        running_loss = 0.0

        for i, (inputs, labels) in enumerate(dataloader):
            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss =self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(dataloader)
        print(f'Epoch [{epoch+1}/{self.num_epochs}], Loss: {epoch_loss:.4f}')

      return self

    def predict(self, X):
      X = torch.tensor(X)
      self.model.eval()
      with torch.no_grad():
        outputs = self.model(X)
        probabilities = F.softmax(outputs, dim=1)  # Get probabilities
        _, predicted = torch.max(outputs.data, 1)
        return predicted.cpu().numpy()

    def predict_proba(self, X):
      X = torch.tensor(X)
      self.model.eval()
      with torch.no_grad():
          # text_input_ids, numerical_features = torch.tensor(X[0]).to(self.device), torch.tensor(X[1]).to(self.device)
          outputs = self.model(X)  # Assuming attention_mask is None
          probabilities = F.softmax(outputs, dim=1)
      return probabilities.numpy()

## With GridSearchCv

In [ ]:
param_grid = {
    'hidden_dim': [64, 128],
    'num_hidden_layers': [1, 2, 3],
    'lr': [1e-4, 1e-3],
    'num_epochs': [5, 10],
    'batch_size': [16, 32]
}

param_grid_two = {
    'hidden_dim': [64, 128, 256],
    'num_hidden_layers': [1, 2, 3, 4],
    'lr': [1e-2, 1e-3, 1e-4, 1e-5],
    'num_epochs': [5, 10, 12],
    'batch_size': [16, 32, 64]
}

In [ ]:
# Define the scoring dictionary
scoring = {
    'accuracy': 'accuracy',
    'precision_macro': make_scorer(precision_score, average='macro', zero_division = 0),
    'recall_macro': make_scorer(recall_score, average='macro', zero_division = 0),
    'f1_macro': make_scorer(f1_score, average='macro', zero_division = 0),
    'precision_weighted': make_scorer(precision_score, average='weighted', zero_division = 0),
    'recall_weighted': make_scorer(recall_score, average='weighted', zero_division = 0),
    'f1_weighted': make_scorer(f1_score, average='weighted', zero_division = 0),
}

In [ ]:
model = Estimator()
grid_search = GridSearchCV(model, param_grid, cv=5, scoring=scoring, refit='f1_weighted')

In [ ]:
print(combined_embeddings.shape)
print(labels.shape)

torch.Size([860, 896])
torch.Size([860])


In [ ]:
combined_embeddings = combined_embeddings.detach().numpy()
labels = labels.long().detach().numpy()

In [ ]:
grid_search.fit(combined_embeddings, labels)

Streaming output truncated to the last 5000 lines.
Epoch [8/10], Loss: 1.2910
Epoch [9/10], Loss: 1.2788
Epoch [10/10], Loss: 1.2274
Epoch [1/12], Loss: 1.6460
Epoch [2/12], Loss: 1.4641
Epoch [3/12], Loss: 1.4152
Epoch [4/12], Loss: 1.3923
Epoch [5/12], Loss: 1.3656
Epoch [6/12], Loss: 1.3222
Epoch [7/12], Loss: 1.2915
Epoch [8/12], Loss: 1.2597
Epoch [9/12], Loss: 1.2267
Epoch [10/12], Loss: 1.1818
Epoch [11/12], Loss: 1.1449
Epoch [12/12], Loss: 1.1236
Epoch [1/12], Loss: 1.6683
Epoch [2/12], Loss: 1.4807
Epoch [3/12], Loss: 1.4390
Epoch [4/12], Loss: 1.4237
Epoch [5/12], Loss: 1.4075
Epoch [6/12], Loss: 1.3842
Epoch [7/12], Loss: 1.3335
Epoch [8/12], Loss: 1.2864
Epoch [9/12], Loss: 1.2752
Epoch [10/12], Loss: 1.2286
Epoch [11/12], Loss: 1.1685
Epoch [12/12], Loss: 1.1440
Epoch [1/12], Loss: 1.6895
Epoch [2/12], Loss: 1.4786
Epoch [3/12], Loss: 1.4601
Epoch [4/12], Loss: 1.4146
Epoch [5/12], Loss: 1.3977
Epoch [6/12], Loss: 1.3622
Epoch [7/12], Loss: 1.3283
Epoch [8/12], Loss: 1.28

KeyboardInterrupt: 

In [ ]:
best_params = grid_search.best_params_
print(best_params)

In [ ]:
# print(torch.isnan(combined_embeddings).any())
# print(torch.isnan(labels).any())

In [ ]:
print("Best score: ", grid_search.best_score_)

In [ ]:
# Get all evaluation results
results = grid_search.cv_results_
print(results)

In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
# save the model
filename = 'event_type_model.pkl'
pickle.dump(best_model, open(filename, 'wb'))

## With K-fold

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = Estimator(input_dim=896, hidden_dim=128, output_dim=8, num_epochs=10, lr=0.001, batch_size=16, num_hidden_layers=1)

accuracies = []
precisions_macro = []
recalls_macro = []
f1s_macro = []
precisions_weighted = []
recalls_weighted = []
f1s_weighted = []

for train_index, val_index in kf.split(combined_embeddings):
    X_train, X_val = combined_embeddings[train_index], combined_embeddings[val_index]
    y_train, y_val = labels[train_index], labels[val_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    accuracy = accuracy_score(y_val, y_pred)
    precision_macro = precision_score(y_val, y_pred, average='macro')
    recall_macro = recall_score(y_val, y_pred, average='macro')
    f1_macro = f1_score(y_val, y_pred, average='macro')
    precision_weighted = precision_score(y_val, y_pred, average='weighted')
    recall_weighted = recall_score(y_val, y_pred, average='weighted')
    f1_weighted = f1_score(y_val, y_pred, average='weighted')

    accuracies.append(accuracy)
    precisions_macro.append(precision_macro)
    recalls_macro.append(recall_macro)
    f1s_macro.append(f1_macro)
    precisions_weighted.append(precision_weighted)
    recalls_weighted.append(recall_weighted)
    f1s_weighted.append(f1_weighted)

    print(f'Fold accuracy: {accuracy}')
    print(f'Fold precision (macro): {precision_macro}')
    print(f'Fold recall (macro): {recall_macro}')
    print(f'Fold F1 score (macro): {f1_macro}')
    print(f'Fold precision (weighted): {precision_weighted}')
    print(f'Fold recall (weighted): {recall_weighted}')
    print(f'Fold F1 score (weighted): {f1_weighted}')

average_accuracy = np.mean(accuracies)
average_precision_macro = np.mean(precisions_macro)
average_recall_macro = np.mean(recalls_macro)
average_f1_macro = np.mean(f1s_macro)
average_precision_weighted = np.mean(precisions_weighted)
average_recall_weighted = np.mean(recalls_weighted)
average_f1_weighted = np.mean(f1s_weighted)

print(f'Average accuracy: {average_accuracy}')
print(f'Average precision (macro): {average_precision_macro}')
print(f'Average recall (macro): {average_recall_macro}')
print(f'Average F1 score (macro): {average_f1_macro}')
print(f'Average precision (weighted): {average_precision_weighted}')
print(f'Average recall (weighted): {average_recall_weighted}')
print(f'Average F1 score (weighted): {average_f1_weighted}')

<ipython-input-20-f9147f84ea98>:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X)
<ipython-input-20-f9147f84ea98>:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y)


Epoch [1/10], Loss: 1.4062
Epoch [2/10], Loss: 1.2679
Epoch [3/10], Loss: 1.2051
Epoch [4/10], Loss: 1.1562
Epoch [5/10], Loss: 1.1255
Epoch [6/10], Loss: 1.1012
Epoch [7/10], Loss: 1.0697
Epoch [8/10], Loss: 1.0441
Epoch [9/10], Loss: 1.0210
Epoch [10/10], Loss: 0.9860
Fold accuracy: 0.600609756097561
Fold precision (macro): 0.3462404870624049
Fold recall (macro): 0.2956074368654326
Fold F1 score (macro): 0.3062520719831488
Fold precision (weighted): 0.5832627055722612
Fold recall (weighted): 0.600609756097561
Fold F1 score (weighted): 0.5684719507590361


<ipython-input-20-f9147f84ea98>:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X)
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
<ipython-input-20-f9147f84ea98>:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTe

Epoch [1/10], Loss: 1.0194
Epoch [2/10], Loss: 0.9919
Epoch [3/10], Loss: 0.9636
Epoch [4/10], Loss: 0.9331
Epoch [5/10], Loss: 0.9204
Epoch [6/10], Loss: 0.8875
Epoch [7/10], Loss: 0.8681
Epoch [8/10], Loss: 0.8411
Epoch [9/10], Loss: 0.8126
Epoch [10/10], Loss: 0.7945
Fold accuracy: 0.6382113821138211
Fold precision (macro): 0.5548805440411078
Fold recall (macro): 0.3599389905332494
Fold F1 score (macro): 0.3877669987387901
Fold precision (weighted): 0.6279808358914913
Fold recall (weighted): 0.6382113821138211
Fold F1 score (weighted): 0.6096919353713522


<ipython-input-20-f9147f84ea98>:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X)
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
<ipython-input-20-f9147f84ea98>:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTe

...

<ipython-input-20-f9147f84ea98>:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X)
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
<ipython-input-20-f9147f84ea98>:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTe

Epoch [1/10], Loss: 0.6909
Epoch [2/10], Loss: 0.6627
Epoch [3/10], Loss: 0.6377
Epoch [4/10], Loss: 0.6017
Epoch [5/10], Loss: 0.5784
Epoch [6/10], Loss: 0.5607
Epoch [7/10], Loss: 0.5433
Epoch [8/10], Loss: 0.5158
Epoch [9/10], Loss: 0.4921
Epoch [10/10], Loss: 0.4715
Fold accuracy: 0.7642276422764228
Fold precision (macro): 0.7978018211181457
Fold recall (macro): 0.5584469666790851
Fold F1 score (macro): 0.6246516964547262
Fold precision (weighted): 0.7859453513722775
Fold recall (weighted): 0.7642276422764228
Fold F1 score (weighted): 0.7533279012237567


<ipython-input-20-f9147f84ea98>:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X)
<ipython-input-20-f9147f84ea98>:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X)
<ipython-input-20-f9147f84ea98>:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y)


Epoch [1/10], Loss: 0.5490
Epoch [2/10], Loss: 0.5253
Epoch [3/10], Loss: 0.4956
Epoch [4/10], Loss: 0.4692
Epoch [5/10], Loss: 0.4474
Epoch [6/10], Loss: 0.4285
Epoch [7/10], Loss: 0.4169
Epoch [8/10], Loss: 0.3921
Epoch [9/10], Loss: 0.3819
Epoch [10/10], Loss: 0.3720
Fold accuracy: 0.8555442522889115
Fold precision (macro): 0.8894807006005355
Fold recall (macro): 0.7367497698380052
Fold F1 score (macro): 0.7894152126909763
Fold precision (weighted): 0.8567623595467992
Fold recall (weighted): 0.8555442522889115
Fold F1 score (weighted): 0.8516472838335941
Average accuracy: 0.7119625089943676
Average precision (macro): 0.6168206746681811
Average recall (macro): 0.47089703893987556
Average F1 score (macro): 0.5054813762164995
Average precision (weighted): 0.7067610371955195
Average recall (weighted): 0.7119625089943676
Average F1 score (weighted): 0.6921102642220067


<ipython-input-20-f9147f84ea98>:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X)


In [ ]:
filename = 'best_event_type_model.pkl'
pickle.dump(model, open(filename, 'wb'))

In [ ]:
test_embed = combined_embeddings[0:5]

In [ ]:
model.predict(test_embed)

<ipython-input-20-f9147f84ea98>:41: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X)


array([0, 1, 1, 3, 0])